<a href="https://colab.research.google.com/github/aiwithpavansama/Stages-of-learning-LLM-Data-pre-processing-from-Scratch-LLM-Base-/blob/main/Stage_1_Data_preprossing_from_Scratch(LLM_Base).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **#reading the words**

In [2]:
!pip install pypdf
from pypdf import PdfReader

In [3]:
# reading the text from the PDF and then getting first 100 characters

reader = PdfReader("/content/drive/MyDrive/The_Verdict.pdf")   # or "/path/to/file.pdf"

raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() or ""

print("total number of characters",len(raw_text))
print(raw_text[:99])

total number of characters 21813
1
The Verdict
Edith Wharton
1908
Exported from Wikisource on May 20, 20242
I HAD always thought Jac


In [4]:
# splitting the words to create tokens

In [5]:
import re
result=re.split(r'(\s)',raw_text)
result = [item for item in result if item.strip()]
print(len(result))
print(result[:99])

3848
['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20,', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius--', 'though', 'a', 'good', 'fellow', 'enough--so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that,', 'in', 'the', 'height', 'of', 'his', 'glory,', 'he', 'had', 'dropped', 'his', 'painting,', 'married', 'a', 'rich', 'widow,', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera.', '(Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence.)', '"The', 'height', 'of', 'his', 'glory"--that', 'was', 'what', 'the', 'women', 'called', 'it.', 'I', 'can', 'hear', 'Mrs.', 'Gideon', 'Thwing--his', 'last', 'Chicago', 'sitter--', 'deploring', 'his', 'unaccountable', 'abdication.', '"Of', 'course']


In [6]:
# we would have created a token on this one but we cannpt since we need camma and other special characters to be as a token so we further split the data so that orginal content is not missed

In [7]:
preprocessed=re.split(r'([,.:;?_!()\']|--|\s)',raw_text)
preprocessed=[item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20', ',', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough']


In [8]:
print(len(preprocessed))

4845


# **Step 2: Creating a Token ID's**:

In [9]:
# sorting in order and giving the token id's
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print("total number of unique words",vocab_size)

total number of unique words 1279


In [10]:
vocab={token:Integer for Integer,token in enumerate(all_words)}

In [11]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('!', 0)
('"', 1)
('"10', 2)
('"19', 3)
('"Ah', 4)
('"Be', 5)
('"Begin', 6)
('"By', 7)
('"Come', 8)
('"Destroyed', 9)
('"Don', 10)
('"Gisburns"', 11)
('"Grindles', 12)
('"Hang', 13)
('"Has', 14)
('"How', 15)
('"I', 16)
('"If', 17)
('"It', 18)
('"Jack', 19)
('"Money', 20)
('"Moon-dancers"', 21)
('"Mr', 22)
('"Mrs', 23)
('"My', 24)
('"Never', 25)
('"Of', 26)
('"Oh', 27)
('"Once', 28)
('"Only', 29)
('"Or', 30)
('"That', 31)
('"The', 32)
('"Then', 33)
('"There', 34)
('"This', 35)
('"We', 36)
('"Well', 37)
('"What', 38)
('"When', 39)
('"Why', 40)
('"Yes', 41)
('"You', 42)
('"but', 43)
('"deadening', 44)
('"dragged', 45)
('"effects"', 46)
('"interesting"', 47)
('"lift', 48)
('"obituary"', 49)
('"strongest', 50)


# **#implement Tokenizer class for encode and decode**

In [31]:
import re

class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, text):
        if isinstance(text, str):
            # Tokenize on spaces, punctuation, and special tokens
            tokens = re.split(r'([,.:;?_!()\']|--|\s|<\|endoftext\|>|<\|unk\|>)', text)
            tokens = [item for item in tokens if item and item.strip()]
        else:
            tokens = text

        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
            else:
                # If <|unk|> is not configured in vocab, skip the unknown token
                pass
        return encoded_ids

    def decode(self, token_ids):
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Initialize the upgraded tokenizer using the vocabulary with special tokens
tokenizer = SimpleTokenizer(vocab=vocab_with_specials)

# Test with 'hello' and '<|endoftext|>'
test_text = "hello It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>"
ids = tokenizer.encode(test_text)
print("Encoded IDs (first ID is <|unk|>):", ids)
print("Decoded output:", tokenizer.decode(ids))

Encoded IDs (first ID is <|unk|>): [1279, 127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927, 1280]
Decoded output: <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>


### Upgraded Tokenizer with Special Tokens Support

We will now recreate our vocabulary list and include two special tokens:
* `<|unk|>`: Used to represent any word that was not present in our training corpus (like `'hello'`).
* `<|endoftext|>`: Used to mark boundaries between different documents or sections.

In [47]:
# 1. Create a new sorted list of unique words and append the special tokens
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])

# 2. Build the updated vocabulary mapping
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

print("New vocab size:", len(vocab_with_specials))
print("ID of <|unk|>:", vocab_with_specials["<|unk|>"])
print("ID of <|endoftext|>:", vocab_with_specials["<|endoftext|>"])

New vocab size: 1281
ID of <|unk|>: 1279
ID of <|endoftext|>: 1280


In [48]:
import re

class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, text):
        if isinstance(text, str):
            # Tokenize on spaces, punctuation, and special tokens
            tokens = re.split(r'([,.:;?_!()\']|--|\s|<\|endoftext\|>|<\|unk\|>)', text)
            tokens = [item for item in tokens if item and item.strip()]
        else:
            tokens = text

        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
            else:
                # If <|unk|> is not configured in vocab, skip the unknown token
                pass
        return encoded_ids

    def decode(self, token_ids):
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Initialize the upgraded tokenizer using the vocabulary with special tokens
tokenizer = SimpleTokenizer(vocab=vocab_with_specials)

In [49]:
# Test with 'hello' and '<|endoftext|>'
test_text = "hello It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>"
ids = tokenizer.encode(test_text)
print("Encoded IDs (first ID is <|unk|>):", ids)
print("Decoded output:", tokenizer.decode(ids))

Encoded IDs (first ID is <|unk|>): [1279, 127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927, 1280]
Decoded output: <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>


In [43]:
# 1. Create a new sorted list of unique words and append the special tokens
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])

# 2. Build the updated vocabulary mapping
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

print("New vocab size:", len(vocab_with_specials))
print("ID of <|unk|>:", vocab_with_specials["<|unk|>"])
print("ID of <|endoftext|>:", vocab_with_specials["<|endoftext|>"])

New vocab size: 1281
ID of <|unk|>: 1279
ID of <|endoftext|>: 1280


In [35]:
tokenizer=SimpleTokenizer(vocab)

In [53]:
text =""" Pavan Kalyan It was not till three years later that, in the course of a few
weeks' idling on the Riviera, it suddenly occurred to me to
wonder why Gisburn had given up his painting. On
reflection, it really was a tempting problem """

In [54]:
ids=tokenizer.encode(text)
print(ids)

[1279, 1279, 127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927]


In [55]:
tokenizer.decode(ids)

"<|unk|> <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem"